# Notebook 04 — Market Basket Analysis: FP-Growth

**Fase 2 · Minilab EduBI · Data Mining**

---

## Tujuan
Menemukan pola **asosiasi jenis produk** yang sering dibeli bersama
menggunakan algoritma **FP-Growth**.

Hasil analisis membantu strategi **cross-selling**: jika pelanggan membeli
jenis produk A, rekomendasikan jenis B yang sering dibeli bersamaan.

### Unit Analisis
- **Basket** = semua pembelian seorang **pelanggan** (bukan per-order)
- Karena setiap order hanya berisi 1 produk, unit basket menggunakan
  `customer_id` agar setiap keranjang berisi berbagai jenis produk
- **Item** = jenis produk (`product_type`) yang di-derive dari `product_name`
  — menghasilkan 9–10 jenis berbeda (Laptop, Monitor, Keyboard, Mouse,
  Audio, Storage, Memory, Webcam, Peripheral, Printer)
- Menggunakan `product_type` (bukan `category` yang hanya 3 nilai)
  agar pola asosiasi lebih kaya dan bermakna secara bisnis

---
## 1. Setup & Koneksi

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import clickhouse_connect
import mlflow

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

CH_HOST    = os.getenv('CH_HOST', 'localhost')
CH_PORT    = int(os.getenv('CH_PORT', 8123))
CH_USER    = os.getenv('CH_USER', 'default')
CH_PASS    = os.getenv('CH_PASSWORD', '')
MLFLOW_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('04_association_market_basket')

client = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT,
    username=CH_USER, password=CH_PASS
)
print('Koneksi ClickHouse berhasil.')

---
## 2. Load Data Transaksi

In [ ]:
query = """
SELECT
    order_id,
    customer_id,
    product_name,
    category,
    quantity,
    toFloat64(total_price) AS total_price,
    branch
FROM silver.silver_sales
WHERE status = 'done'
ORDER BY order_id
"""

df = client.query_df(query)

# ─── Derive product_type dari product_name ────────────────────────────────────
def get_product_type(name: str) -> str:
    n = name.lower()
    if any(k in n for k in ['laptop', 'macbook', 'surface pro']):
        return 'Laptop'
    if 'monitor' in n:
        return 'Monitor'
    if 'printer' in n:
        return 'Printer'
    if 'tablet' in n or 'ipad' in n:
        return 'Tablet'
    if 'webcam' in n:
        return 'Webcam'
    if 'mouse pad' in n or 'cooling pad' in n:
        return 'Peripheral'
    if 'mouse' in n:
        return 'Mouse'
    if 'keyboard' in n:
        return 'Keyboard'
    if any(k in n for k in ['headset', 'headphone', 'speaker']):
        return 'Audio'
    if any(k in n for k in ['ssd', 'flash disk']):
        return 'Storage'
    if 'ram' in n:
        return 'Memory'
    if any(k in n for k in ['hub', 'usb-c', 'charger', 'power bank']):
        return 'Peripheral'
    return 'Lainnya'

df['product_type'] = df['product_name'].apply(get_product_type)

print(f'Total transaksi  : {len(df)}')
print(f'Order unik       : {df["order_id"].nunique()}')
print(f'Customer unik    : {df["customer_id"].nunique()}')
print(f'Jenis produk     : {sorted(df["product_type"].unique().tolist())}')
print(f'\nDistribusi product_type:')
print(df["product_type"].value_counts())
df.head()

---
## 3. Eksplorasi Distribusi Produk & Kategori

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 produk
top_products = df.groupby('product_name')['order_id'].count().sort_values(ascending=False).head(10)
top_products.plot.barh(ax=ax1, color='steelblue')
ax1.set(title='Top 10 Produk Terlaris', xlabel='Jumlah Order')
ax1.invert_yaxis()

# Distribusi product_type
type_dist = df.groupby('product_type')['order_id'].count().sort_values(ascending=False)
type_dist.plot.bar(ax=ax2, color='coral', rot=30)
ax2.set(title='Distribusi Order per Jenis Produk', ylabel='Jumlah Order')

plt.tight_layout()
plt.savefig('experiments/mba_product_distribution.png', dpi=100)
plt.show()

---
## 4. Buat Transaction Matrix

In [ ]:
# Kelompokkan product_type per PELANGGAN
# Karena setiap order hanya 1 produk, unit basket = customer_id
# Item = product_type (10 jenis) agar pola lebih kaya dari sekedar 3 kategori
basket = df.groupby('customer_id')['product_type'].apply(list).reset_index()
transactions = basket['product_type'].tolist()

print(f'Total basket (pelanggan) : {len(transactions)}')
print(f'Rata-rata item per basket: {sum(len(t) for t in transactions)/len(transactions):.1f}')
print(f'Min item                 : {min(len(t) for t in transactions)}')
print(f'Max item                 : {max(len(t) for t in transactions)}')
print()
print('Contoh basket 3 pelanggan pertama:')
for i, (idx, row) in enumerate(basket.head(3).iterrows()):
    print(f'  {row["customer_id"]}: {sorted(set(row["product_type"]))}')

# Encode ke format boolean matrix (deduplicate otomatis oleh TransactionEncoder)
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(f'\nDimensi matrix   : {df_encoded.shape}')
print('Frekuensi jenis produk per pelanggan:')
print(df_encoded.sum().sort_values(ascending=False).to_string())
df_encoded.head()

---
## 5. FP-Growth: Frequent Itemsets

In [ ]:
# Dengan 20 pelanggan dan 10 jenis produk:
# MIN_SUPPORT=0.15 → pola muncul di minimal 3 pelanggan (3/20)
# Lebih realistis untuk dataset 20 pelanggan
MIN_SUPPORT = 0.15

frequent_itemsets = fpgrowth(
    df_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True
)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values(['length', 'support'], ascending=[True, False])

print(f'Frequent itemsets ditemukan: {len(frequent_itemsets)}')
print()
for length in sorted(frequent_itemsets['length'].unique()):
    subset = frequent_itemsets[frequent_itemsets['length'] == length]
    print(f'  {length}-itemset ({len(subset)} buah):')
    for _, row in subset.iterrows():
        print(f'    {set(row["itemsets"])} — support: {row["support"]:.0%}')

---
## 6. Aturan Asosiasi (Association Rules)

In [ ]:
MIN_CONFIDENCE = 0.5
MIN_LIFT       = 1.0

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=MIN_CONFIDENCE
)
rules = rules[rules['lift'] >= MIN_LIFT].sort_values('lift', ascending=False)

# Format tampilan
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))

print(f'Aturan asosiasi ditemukan: {len(rules)}')
if len(rules) > 0:
    print()
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string(index=False))


In [ ]:
# Visualisasi Support vs Confidence (ukuran bubble = lift)
if len(rules) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(
        rules['support'],
        rules['confidence'],
        c=rules['lift'],
        s=rules['lift'] * 80,
        cmap='YlOrRd',
        alpha=0.7,
        edgecolors='gray'
    )
    plt.colorbar(scatter, ax=ax, label='Lift')
    ax.set(xlabel='Support', ylabel='Confidence',
           title='Association Rules: Support vs Confidence (ukuran = Lift)')
    for _, row in rules.head(5).iterrows():
        ax.annotate(
            f"{row['antecedents']} → {row['consequents']}",
            (row['support'], row['confidence']),
            textcoords='offset points', xytext=(5, 5), fontsize=8
        )
    plt.tight_layout()
    plt.savefig('experiments/mba_rules_scatter.png', dpi=100)
    plt.show()
else:
    print('Tidak ada aturan asosiasi yang ditemukan. Coba kurangi min_support atau min_confidence.')

---
## 7. Log Hasil ke MLflow

In [ ]:
with mlflow.start_run(run_name='fpgrowth_product_type'):
    mlflow.log_param('min_support',    MIN_SUPPORT)
    mlflow.log_param('min_confidence', MIN_CONFIDENCE)
    mlflow.log_param('min_lift',       MIN_LIFT)
    mlflow.log_param('item_level',     'product_type')
    mlflow.log_param('basket_unit',    'customer_id')

    mlflow.log_metric('n_frequent_itemsets', len(frequent_itemsets))
    mlflow.log_metric('n_rules',             len(rules))
    if len(rules) > 0:
        mlflow.log_metric('max_lift',        rules['lift'].max())
        mlflow.log_metric('avg_confidence',  rules['confidence'].mean())
        mlflow.log_artifact('experiments/mba_rules_scatter.png')

    mlflow.log_artifact('experiments/mba_product_distribution.png')

    # Simpan rules ke CSV
    rules_path = 'experiments/association_rules.csv'
    rules.to_csv(rules_path, index=False)
    mlflow.log_artifact(rules_path)

    print(f'Log berhasil.')
    print(f'  Frequent itemsets : {len(frequent_itemsets)}')
    print(f'  Association rules : {len(rules)}')

---
## 8. Interpretasi Bisnis

In [ ]:
if len(rules) > 0:
    print('=== TOP 5 REKOMENDASI CROSS-SELLING ===')
    print()
    for _, row in rules.head(5).iterrows():
        print(f'Jika pelanggan membeli : {row["antecedents"]}')
        print(f'Rekomendasikan         : {row["consequents"]}')
        print(f'Confidence: {row["confidence"]:.1%}  |  Lift: {row["lift"]:.2f}  |  Support: {row["support"]:.1%}')
        print('-' * 60)

---
## 9. Kesimpulan

**Pertanyaan Diskusi:**
1. Apa perbedaan antara algoritma Apriori dan FP-Growth? Mengapa FP-Growth lebih efisien?
2. Lift = 1 artinya apa? Kapan sebuah aturan asosiasi dianggap bermakna?
3. Jika data transaksi sangat jarang (sparse), bagaimana efeknya terhadap min_support?
4. Bagaimana aturan asosiasi yang ditemukan dapat diimplementasikan di platform e-commerce?

**Lihat hasil eksperimen di MLflow:** http://localhost:5000